# Implementação da GNN *(em construção)*

In [ ]:
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU
from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader
from torch_geometric.nn import HeteroConv, GINEConv, global_mean_pool

In [ ]:
# Importação do dataset
def gerar_grafo_heterogeneo(classe_alvo):
    return
    ###### IMPLEMENTAR IMPORTAÇÃO DO GRAFO ######
    data = HeteroData()
    
    num_origem = torch.randint(3, 7, (1,)).item()
    num_destino = torch.randint(3, 7, (1,)).item()
    
    # Nós 'origem' e 'destino', cada um com 3 atributos: [data_registro, é_aluguel, nome_tipo]
    data['origem'].x = torch.randn((num_origem, 3))
    data['destino'].x = torch.randn((num_destino, 3))
    
    # Arestas de tipo 'mudança': origem -> destino com 2 atributos: [variacao_despesas, distância]
    src_mudanca = torch.randint(0, num_origem, (6,))
    dst_mudanca = torch.randint(0, num_destino, (6,))
    data['origem', 'mudanca', 'destino'].edge_index = torch.stack([src_mudanca, dst_mudanca], dim=0)
    data['origem', 'mudanca', 'destino'].edge_attr = torch.randn((6, 2))
    
    # Arestas de tipo 'vizinhança': origem -> destino com 1 atributo: [distância]
    src_viz = torch.randint(0, num_origem, (5,))
    dst_viz = torch.randint(0, num_destino, (5,))
    data['origem', 'vizinhanca', 'destino'].edge_index = torch.stack([src_viz, dst_viz], dim=0)
    data['origem', 'vizinhanca', 'destino'].edge_attr = torch.randn((5, 1))
    
    # Rótulo da classe do grafo inteiro: 0 ("not_eviction") ou 1 ("eviction")
    data.y = torch.tensor([classe_alvo], dtype=torch.long)
    
    return data


dataset = [gerar_grafo_heterogeneo(torch.randint(0, 2, (1,)).item()) for _ in range(40)]
train_loader = DataLoader(dataset[:30], batch_size=8, shuffle=True)

In [ ]:
# Definição da arquitetura da GNN
class HeteroGraphClassifier(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels, metadata):
        super().__init__()

        ##### DEFINIR NÚMERO DE CAMADAS GINECONV #######
        
        convs1 = {}
        convs2 = {}
        
        # Configura camadas convolucionais personalizadas para cada tipo de aresta
        for edge_type in metadata[1]:
            rel = edge_type[1]
            edge_dim = 2 if rel == 'mudanca' else 1
            
            # Primeira camada
            nn1 = Sequential(
                Linear(3, hidden_channels), 
                ReLU(), 
                Linear(hidden_channels, hidden_channels)
            )
            convs1[edge_type] = GINEConv(nn1, edge_dim=edge_dim)
            
            # Segunda camada
            nn2 = Sequential(
                Linear(hidden_channels, hidden_channels), 
                ReLU(), 
                Linear(hidden_channels, hidden_channels)
            )
            convs2[edge_type] = GINEConv(nn2, edge_dim=edge_dim)

        ###### DEFINIR NÚMERO DE CAMADAS HETEROCONV ######
        self.conv1 = HeteroConv(convs1, aggr='sum')
        self.conv2 = HeteroConv(convs2, aggr='sum')
        
        # Classificador linear
        self.lin = Linear(hidden_channels, out_channels)

    def forward(self, x_dict, edge_index_dict, edge_attr_dict, batch_dict):
        # 1. Convoluções heterogêneas considerando os atributos de nós e arestas
        x_dict = self.conv1(x_dict, edge_index_dict, edge_attr_dict=edge_attr_dict)
        x_dict = {key: F.relu(x) for key, x in x_dict.items()}
        
        x_dict = self.conv2(x_dict, edge_index_dict, edge_attr_dict=edge_attr_dict)
        x_dict = {key: F.relu(x) for key, x in x_dict.items()}
        
        # 2. Pooling global separado por tipo de nó e agregação final do grafo
        pooled = []
        for node_type, x in x_dict.items():
            b = batch_dict[node_type]
            pooled.append(global_mean_pool(x, b))
            
        # Combina as representações globais dos tipos de nós
        x_graph = torch.stack(pooled, dim=0).sum(dim=0)
        
        # 3. Classificação
        x_graph = F.dropout(x_graph, p=0.5, training=self.training)
        out = self.lin(x_graph)
        return out

# Instanciação do modelo baseada no metadata do grafo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
metadata = dataset[0].metadata()

model = HeteroGraphClassifier(hidden_channels=16, out_channels=2, metadata=metadata).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
# Treinamento
model.train()
for epoch in range(10):
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        # O DataLoader do PyG gerencia automaticamente os dicionários heterogêneos e lotes
        out = model(
            batch.x_dict, 
            batch.edge_index_dict, 
            batch.edge_attr_dict, 
            batch.batch_dict
        )

        ###### DEFINIR FUNÇÃO DE PERDA (CUSTOMIZAR?) ######
        loss = criterion(out, batch.y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch.num_graphs
        
    epoch_loss = total_loss / len(train_loader.dataset)
    print(f"Época {epoch+1:02d} | Loss: {epoch_loss:.4f}")